In [1]:
import numpy as np
import pandas as pd
import optuna
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("All imports done!")

All imports done!


In [2]:
df = pd.read_csv("Dataset_B.csv")

targets = ['r1','r2','r3','D1','D2','D3']
feature_cols = [c for c in df.columns if c not in targets]

X = df[feature_cols].values
Y = df[targets].values

# Handle NaNs
col_means = np.nanmean(X, axis=0)
inds = np.where(np.isnan(X))
X[inds] = np.take(col_means, inds[1])

# Remove zero variance
stds = X.std(axis=0)
mask = stds > 1e-12
X = X[:, mask]

# Normalize
from sklearn.preprocessing import StandardScaler
scaler_X = StandardScaler()
scaler_Y = StandardScaler()
Xn = scaler_X.fit_transform(X)
Yn = scaler_Y.fit_transform(Y)

# Split
X_train, X_test, Y_train, Y_test = train_test_split(
    Xn, Yn, test_size=0.2, random_state=42
)

print("Data loaded!")
print("X shape:", X_train.shape)
print("Y shape:", Y_train.shape)

Data loaded!
X shape: (3918, 20)
Y shape: (3918, 6)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9308\1010420742.py:10: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(X, axis=0)


In [3]:
#Bayesian Tuning For XGBoost
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 4, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    
    xgb = XGBRegressor(**params, objective="reg:squarederror", 
                        random_state=42, n_jobs=-1, verbosity=0)
    model = MultiOutputRegressor(xgb)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    r2 = r2_score(Y_test, Y_pred)
    return r2

print("Running Bayesian Tuning for XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=20)

print("Best XGBoost params:", study_xgb.best_params)
print("Best R²:", study_xgb.best_value)

Running Bayesian Tuning for XGBoost...
Best XGBoost params: {'n_estimators': 477, 'max_depth': 6, 'learning_rate': 0.02458413474795134, 'subsample': 0.7894816013956796, 'colsample_bytree': 0.6197542907734834}
Best R²: 0.5708642988656326


In [4]:
xgb_best_params = {
    'n_estimators': 477,
    'max_depth': 6,
    'learning_rate': 0.02458,
    'subsample': 0.7895,
    'colsample_bytree': 0.6197
}
print("XGBoost best params saved!")

XGBoost best params saved!


In [5]:
#Bayesian Tuning For RF
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth':    trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
    }
    
    rf = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model = MultiOutputRegressor(rf)
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    r2 = r2_score(Y_test, Y_pred)
    return r2

print("Running Bayesian Tuning for RF...")
study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=20)

print("Best RF params:", study_rf.best_params)
print("Best R²:", study_rf.best_value)

Running Bayesian Tuning for RF...
Best RF params: {'n_estimators': 596, 'max_depth': 17, 'min_samples_split': 5}
Best R²: 0.5760858605464345


In [6]:
rf_best_params = {
    'n_estimators': 596,
    'max_depth': 17,
    'min_samples_split': 5
}
print("RF best params saved!")

RF best params saved!


In [7]:
#Bayesian Tuning For FFNN
def objective_ffnn(trial):
    n_layers = trial.suggest_int('n_layers', 2, 5)
    n_units  = trial.suggest_int('n_units', 64, 512)
    lr       = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    
    layers = []
    in_dim = X_train.shape[1]
    for i in range(n_layers):
        out_dim = n_units
        layers += [nn.Linear(in_dim, out_dim), nn.ReLU()]
        in_dim = out_dim
    layers.append(nn.Linear(in_dim, 6))
    model = nn.Sequential(*layers)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    X_tr = torch.tensor(X_train, dtype=torch.float32)
    Y_tr = torch.tensor(Y_train, dtype=torch.float32)
    X_te = torch.tensor(X_test,  dtype=torch.float32)
    
    for epoch in range(200):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_tr), Y_tr)
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        Y_pred = model(X_te).numpy()
    
    r2 = r2_score(Y_test, Y_pred)
    return r2

print("Running Bayesian Tuning for FFNN...")
study_ffnn = optuna.create_study(direction='maximize')
study_ffnn.optimize(objective_ffnn, n_trials=20)

print("Best FFNN params:", study_ffnn.best_params)
print("Best R²:", study_ffnn.best_value)

Running Bayesian Tuning for FFNN...
Best FFNN params: {'n_layers': 3, 'n_units': 206, 'lr': 0.0035931728736726992}
Best R²: 0.6491015850233555


In [8]:
ffnn_best_params = {
    'n_layers': 3,
    'n_units': 206,
    'lr': 0.003599
}
print("FFNN best params saved!")

FFNN best params saved!


In [9]:
print("========= BEST PARAMETERS =========")
print("\nXGBoost:")
for k,v in study_xgb.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best R²: {study_xgb.best_value:.4f}")

print("\nRandom Forest:")
for k,v in study_rf.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best R²: {study_rf.best_value:.4f}")

print("\nFFNN:")
for k,v in study_ffnn.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best R²: {study_ffnn.best_value:.4f}")

========= BEST PARAMETERS =========

XGBoost:
  n_estimators: 477
  max_depth: 6
  learning_rate: 0.02458413474795134
  subsample: 0.7894816013956796
  colsample_bytree: 0.6197542907734834
  Best R²: 0.5709

Random Forest:
  n_estimators: 596
  max_depth: 17
  min_samples_split: 5
  Best R²: 0.5761

FFNN:
  n_layers: 3
  n_units: 206
  lr: 0.0035931728736726992
  Best R²: 0.6491
